# 🔗 Clase 1 — Correlación entre variables
## Unidad: Correlación y modelamiento

**Situación:** Trabajas en inteligencia comercial de una empresa de servicios de internet. La gerencia detectó una alta tasa de abandono (churn) y quiere tomar decisiones basadas en datos **antes** de lanzar una campaña de fidelización.

**Preguntas clave:**
- ¿Existe relación entre satisfacción del cliente y abandono?
- ¿A mayor uso del servicio, se mantiene más tiempo el contrato?
- ¿Es posible anticipar la intención de abandono observando otras variables?

**Objetivos:**
- Visualizar relaciones con **gráficos de dispersión**
- Analizar variables categóricas con **tablas de contingencia**
- Calcular e interpretar el **coeficiente de Pearson**
- Reflexionar sobre **correlación vs. causalidad**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

print('✅ Librerías cargadas')
print(f'pandas {pd.__version__} | seaborn {sns.__version__}')

---
## PARTE 1 — Conceptos: gráfico de dispersión y patrones visuales

### 1.1 Ejemplo introductorio: horas de capacitación vs productividad

In [ ]:
# Ejemplo exacto de la presentación
df_intro = pd.DataFrame({
    'horas_capacitacion': [4, 6, 8, 10, 12, 14, 16],
    'productividad':      [65, 67, 70, 74, 78, 80, 84]
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Relación entre horas de capacitación y productividad', fontweight='bold')

# Scatterplot simple
sns.scatterplot(data=df_intro, x='horas_capacitacion', y='productividad',
                ax=axes[0], s=100, color='#2E75B6')
axes[0].set_title('Gráfico de dispersión')
axes[0].set_xlabel('Horas de capacitación')
axes[0].set_ylabel('Productividad (%)')

# Con línea de tendencia (regplot)
sns.regplot(data=df_intro, x='horas_capacitacion', y='productividad',
            ax=axes[1], ci=None, color='#2E75B6',
            scatter_kws={'s': 80}, line_kws={'color': '#ED7D31', 'linewidth': 2})
axes[1].set_title('Con línea de tendencia (regplot, ci=None)')
axes[1].set_xlabel('Horas de capacitación')
axes[1].set_ylabel('Productividad (%)')

plt.tight_layout()
plt.show()

r = df_intro['horas_capacitacion'].corr(df_intro['productividad'])
print(f'Correlación de Pearson r = {r:.4f} → correlación positiva muy fuerte')

### 1.2 Galería de patrones de correlación visual

In [ ]:
np.random.seed(42)
n = 80

# Generar los 4 tipos de patrones
x_base = np.random.uniform(0, 10, n)
patrones = {
    'Positiva fuerte\n(r ≈ 0.95)':   (x_base, x_base + np.random.normal(0, 0.5, n)),
    'Negativa fuerte\n(r ≈ -0.90)':  (x_base, -x_base + 10 + np.random.normal(0, 0.7, n)),
    'Nula\n(r ≈ 0.0)':               (x_base, np.random.uniform(0, 10, n)),
    'No lineal\n(curva en U)':        (x_base, (x_base - 5)**2 + np.random.normal(0, 0.5, n)),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Tipos de patrones de correlación visual', fontweight='bold')
colores = ['#2E75B6', '#ED7D31', '#A5A5A5', '#70AD47']

for ax, (titulo, (x, y)), color in zip(axes, patrones.items(), colores):
    r_pat = np.corrcoef(x, y)[0, 1]
    ax.scatter(x, y, s=25, alpha=0.6, color=color)
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.text(0.05, 0.92, f'r = {r_pat:.2f}', transform=ax.transAxes,
            fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

### ✏️ Ejercicio 1 — Reflexiona:

In [ ]:
# ✏️ ¿Por qué la correlación de la curva en U es cercana a 0 aunque hay una relación clara?
r_nolineal = ""

# ✏️ ¿Por qué siempre debes visualizar antes de calcular Pearson?
r_visual_primero = ""

# ✏️ Da un ejemplo de correlación negativa de tu contexto laboral:
r_ejemplo_neg = ""

print(f'Curva en U: {r_nolineal}')
print(f'Visualizar primero: {r_visual_primero}')
print(f'Ejemplo negativo: {r_ejemplo_neg}')

---
## PARTE 2 — Tablas de contingencia

### 2.1 Ejemplo de la presentación: tipo de contrato vs abandono

In [ ]:
# Ejemplo exacto de la presentación
df_ex = pd.DataFrame({
    'tipo_contrato': ['mensual','mensual','anual','anual','mensual','anual','mensual'],
    'abandono':      ['sí',     'no',     'no',   'no',   'sí',     'no',   'sí']
})

# Tabla de frecuencias absolutas
tabla = pd.crosstab(df_ex['tipo_contrato'], df_ex['abandono'])
print('=== Tabla de contingencia — Frecuencias absolutas ===')
print(tabla)
print()

# Tabla normalizada por filas (proporciones)
tabla_norm = pd.crosstab(df_ex['tipo_contrato'], df_ex['abandono'], normalize='index').round(2)
print('=== Normalizada por filas (proporciones) ===')
print(tabla_norm)
print()
print('Interpretación:')
print('  Contratos mensuales: 3 de 4 (75%) abandonaron el servicio')
print('  Contratos anuales:   0 de 3 (0%) abandonaron el servicio')
print('  → Podría existir relación entre tipo de contrato y abandono')

In [ ]:
# Con márgenes y visualización
tabla_marg = pd.crosstab(df_ex['tipo_contrato'], df_ex['abandono'], margins=True)
print('=== Con totales marginales ===')
print(tabla_marg)

---
## PARTE 3 — Coeficiente de correlación de Pearson

### 3.1 Ejemplo de la presentación: correos abiertos vs intención de compra

In [ ]:
# Ejemplo exacto de la presentación
df_mkt = pd.DataFrame({
    'correos_abiertos': [1, 3, 5, 7, 8, 10, 12],
    'intencion_compra': [2, 3, 5, 6, 6,  8,  9]
})

print('=== Matriz de correlación ===')
print(df_mkt[['correos_abiertos', 'intencion_compra']].corr().round(3))
print()

r_mkt = df_mkt['correos_abiertos'].corr(df_mkt['intencion_compra'])
print(f'r = {r_mkt:.3f} → correlación positiva muy fuerte')
print('Interpretación: cuando se abren más correos, la intención de compra también aumenta')

### 3.2 Escala de interpretación del coeficiente r

In [ ]:
tabla_r = pd.DataFrame({
    'Valor de r':     ['1.0', '0.70 – 0.99', '0.30 – 0.69', '0.00 – 0.29',
                       '-0.30 – -0.69', '-0.70 – -0.99', '-1.0'],
    'Interpretación': [
        'Correlación positiva perfecta',
        'Correlación positiva fuerte',
        'Correlación positiva moderada',
        'Correlación débil o nula',
        'Correlación negativa moderada',
        'Correlación negativa fuerte',
        'Correlación negativa perfecta'
    ]
})
print(tabla_r.to_string(index=False))

---
## PARTE 4 — Correlación vs. Causalidad

### 4.1 Demostración: correlación espuria

In [ ]:
# Ejemplo clásico: variable confusora (tercera variable)
# A más bomberos → más daño. ¿Los bomberos causan el daño?
np.random.seed(10)
tamano_incendio = np.random.randint(1, 10, 30)
bomberos = tamano_incendio * 3 + np.random.randint(-2, 3, 30)
danio     = tamano_incendio * 15 + np.random.randint(-5, 6, 30)

r_espuria = np.corrcoef(bomberos, danio)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Correlación espuria — Ejemplo: Bomberos vs Daño del incendio',
             fontweight='bold')

axes[0].scatter(bomberos, danio, color='#ED7D31', alpha=0.7, s=60)
axes[0].set_title(f'Bomberos vs Daño (r={r_espuria:.2f}) → ¿Los bomberos causan el daño?')
axes[0].set_xlabel('Cantidad de bomberos')
axes[0].set_ylabel('Daño estimado ($)')

# La variable confusora explica ambas
scatter = axes[1].scatter(bomberos, danio, c=tamano_incendio,
                           cmap='YlOrRd', s=80, alpha=0.8)
plt.colorbar(scatter, ax=axes[1], label='Tamaño del incendio')
axes[1].set_title('Variable confusora: tamaño del incendio explica ambas')
axes[1].set_xlabel('Cantidad de bomberos')
axes[1].set_ylabel('Daño estimado ($)')

plt.tight_layout()
plt.show()

print(f'r(bomberos, daño) = {r_espuria:.2f}  → correlación alta, pero NO hay causalidad')
print('La variable oculta (tamaño del incendio) explica ambas variables.')

### ✏️ Ejercicio 4 — Reflexiona sobre causalidad:

In [ ]:
# ✏️ Para cada par de variables, indica si podría ser correlación sin causalidad y por qué:
casos = {
    'Más quejas → mayor satisfacción (r negativo)': "",
    'Mayor uso de datos → mayor fidelidad':          "",
    'Más vendedores → mayores ventas':               "",
}
print('--- ANÁLISIS CAUSALIDAD vs. CORRELACIÓN ---')
for caso, r in casos.items():
    print(f'\n{caso}')
    print(f'  Respuesta: {r}')

# ✏️ ¿Qué lenguaje usarías para reportar una correlación sin asumir causalidad?
r_lenguaje = ""
print(f'\nLenguaje correcto: {r_lenguaje}')

---
## PARTE 5 — Actividad guiada: Análisis de churn en clientes de servicios digitales

### 5.1 Cargar y diagnosticar el dataset

In [ ]:
df = pd.read_csv('clientes_servicio.csv')

print(f'Registros: {df.shape[0]} | Columnas: {df.shape[1]}')
print()
print(df.head(8))
print()
print('=== Tipos de dato ===')
print(df.dtypes)
print()
print('=== Valores faltantes ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Detectar inconsistencias y limpiar
print('=== Valores únicos en abandono ===')
print(df['abandono'].unique())
print('=== Valores únicos en tipo_contrato ===')
print(df['tipo_contrato'].unique())

# Corregir
df['abandono']       = df['abandono'].str.lower().str.strip()
df['tipo_contrato']  = df['tipo_contrato'].str.lower().str.strip()
df = df.dropna(subset=['satisfaccion', 'uso_mensual_gb'])

print(f'\nRegistros después de limpieza: {len(df)}')
print('abandono único:', df['abandono'].unique())
print('tipo_contrato único:', df['tipo_contrato'].unique())

### 5.2 Visualizar relaciones entre variables numéricas (scatterplots)

In [ ]:
# Scatterplots exactos de la presentación
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Relaciones entre variables numéricas — Clientes', fontweight='bold')

pares = [
    ('uso_mensual_gb', 'satisfaccion',   'Uso mensual vs. Satisfacción'),
    ('tiempo_contrato', 'satisfaccion',  'Tiempo de contrato vs. Satisfacción'),
    ('tiempo_contrato', 'uso_mensual_gb','Tiempo de contrato vs. Uso (GB)'),
]

for ax, (x, y, titulo) in zip(axes, pares):
    sns.scatterplot(data=df, x=x, y=y, hue='abandono',
                    palette={'sí': '#FC4E4E', 'no': '#2E75B6'},
                    alpha=0.6, s=40, ax=ax)
    r_val = df[x].corr(df[y])
    ax.set_title(f'{titulo}\nr = {r_val:.3f}', fontsize=9, fontweight='bold')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(title='Abandono', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Con líneas de tendencia (regplot) por grupo de abandono
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Líneas de tendencia por grupo de abandono', fontweight='bold')

for ax, (x, y, titulo) in zip(axes, pares[:2]):
    for grupo, color in [('no', '#2E75B6'), ('sí', '#FC4E4E')]:
        subset = df[df['abandono'] == grupo]
        sns.regplot(data=subset, x=x, y=y, ci=None, ax=ax, color=color,
                    scatter_kws={'alpha': 0.3, 's': 20},
                    line_kws={'linewidth': 2.5, 'label': f'Abandono {grupo}'})
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 5.3 Calcular el coeficiente de correlación de Pearson

In [ ]:
# Matriz de correlación de todas las variables numéricas
numericas = df[['tiempo_contrato', 'uso_mensual_gb', 'satisfaccion', 'n_reclamos']]
matriz_corr = numericas.corr().round(3)

print('=== Matriz de correlación de Pearson ===')
print(matriz_corr)

# Heatmap
plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(matriz_corr, dtype=bool), k=1)
sns.heatmap(matriz_corr, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title('Heatmap de correlación de Pearson', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Interpretación de cada par
print('=== Interpretación de pares de variables ===')
pares_interp = [
    ('tiempo_contrato', 'satisfaccion'),
    ('uso_mensual_gb', 'satisfaccion'),
    ('n_reclamos', 'satisfaccion'),
    ('tiempo_contrato', 'uso_mensual_gb'),
]

def interpretar_r(r):
    a = abs(r)
    dir_ = 'positiva' if r > 0 else 'negativa'
    if a >= 0.70: fuerza = 'fuerte'
    elif a >= 0.30: fuerza = 'moderada'
    else: fuerza = 'débil o nula'
    return f'Correlación {dir_} {fuerza} (r={r:.3f})'

for v1, v2 in pares_interp:
    r = df[v1].corr(df[v2])
    print(f'  {v1:25} ↔ {v2:20}: {interpretar_r(r)}')

### 5.4 Tablas de contingencia: abandono vs variables categóricas

In [ ]:
# Tabla de contingencia: abandono vs tipo de contrato
tabla_cont = pd.crosstab(df['tipo_contrato'], df['abandono'],
                          margins=True, margins_name='Total')
print('=== Abandono × Tipo de contrato (frecuencias absolutas) ===')
print(tabla_cont)
print()

tabla_prop = pd.crosstab(df['tipo_contrato'], df['abandono'],
                          normalize='index').round(3) * 100
print('=== Proporciones por fila (%) ===')
print(tabla_prop.round(1))

In [ ]:
# Tabla: abandono vs satisfacción alta (> 7) — exactamente como en la presentación
tabla_satis = pd.crosstab(df['abandono'], df['satisfaccion'] > 7)
tabla_satis.columns = ['Satisfacción ≤ 7', 'Satisfacción > 7']
print('=== Abandono × Satisfacción alta (como en la presentación) ===')
print(tabla_satis)
print()

tabla_satis_pct = pd.crosstab(df['abandono'], df['satisfaccion'] > 7, normalize='columns').round(3)*100
tabla_satis_pct.columns = ['Satisfacción ≤ 7', 'Satisfacción > 7']
print('=== Proporciones por columna (%) ===')
print(tabla_satis_pct.round(1))

In [ ]:
# Visualización: tasas de abandono por tipo de contrato y plan
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Tasa de abandono por variables categóricas', fontweight='bold')

for ax, col, titulo in zip(
    axes,
    ['tipo_contrato', 'plan', 'tipo_cliente'],
    ['Tipo de contrato', 'Plan', 'Tipo de cliente']
):
    tasa = df.groupby(col)['abandono'].apply(
        lambda x: (x == 'sí').mean() * 100
    ).sort_values(ascending=False)
    bars = ax.bar(tasa.index, tasa.values, color='#2E75B6', edgecolor='white')
    ax.set_title(f'Tasa de abandono por {titulo}', fontsize=9, fontweight='bold')
    ax.set_ylabel('% abandono')
    ax.set_ylim(0, 80)
    ax.tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, tasa.values):
        ax.text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%',
                ha='center', fontsize=9)

plt.tight_layout()
plt.show()

### 5.5 Pairplot: visión completa de relaciones

In [ ]:
subset_pair = df[['tiempo_contrato','uso_mensual_gb','satisfaccion','n_reclamos','abandono']].copy()

g = sns.pairplot(subset_pair, hue='abandono',
                  palette={'sí': '#FC4E4E', 'no': '#2E75B6'},
                  plot_kws={'alpha': 0.4, 's': 15},
                  diag_kind='kde')
g.figure.suptitle('Pairplot — Relaciones entre variables (coloreado por abandono)',
                   y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

### ✏️ Preguntas de reflexión y cierre:

In [ ]:
# ✏️ 1. ¿Qué relaciones identificaste entre las variables numéricas?
c1 = ""

# ✏️ 2. ¿Qué factores parecen estar más asociados con la permanencia del cliente?
c2 = ""

# ✏️ 3. ¿Puedes afirmar que alguna variable CAUSA el abandono? ¿Por qué sí o no?
c3 = ""

# ✏️ 4. ¿Qué limitaciones tiene este análisis basado solo en correlaciones?
c4 = ""

# ✏️ 5. ¿Qué recomendarías a la gerencia basándote en los datos?
c5 = ""

# Resumen automático para apoyar las reflexiones
print('--- DATOS DE APOYO PARA LAS REFLEXIONES ---')
print(f'Tasa de abandono global:              {(df["abandono"]=="sí").mean()*100:.1f}%')
print(f'r(satisfaccion, tiempo_contrato):     {df["satisfaccion"].corr(df["tiempo_contrato"]):.3f}')
print(f'r(n_reclamos, satisfaccion):          {df["n_reclamos"].corr(df["satisfaccion"]):.3f}')
tipo_tasa = df.groupby('tipo_contrato')['abandono'].apply(lambda x: (x=='sí').mean()*100).round(1)
print(f'Tasa abandono por tipo de contrato:')
print(tipo_tasa.to_string())

print('\n--- CONCLUSIONES ---')
for i, c in enumerate([c1, c2, c3, c4, c5], 1):
    print(f'{i}. {c}')

---
## 📋 Resumen de funciones y buenas prácticas

| Técnica | Función | Cuándo usarla |
|---------|---------|---------------|
| Gráfico de dispersión | `sns.scatterplot(x=, y=, hue=)` | Dos variables numéricas |
| Con línea de tendencia | `sns.regplot(ci=None)` | Confirmar patrón lineal |
| Pairplot completo | `sns.pairplot(hue=)` | Explorar múltiples variables |
| Tabla de contingencia | `pd.crosstab(v1, v2)` | Dos variables categóricas |
| Proporciones | `pd.crosstab(..., normalize='index')` | Comparar tasas entre grupos |
| Correlación par | `df[v1].corr(df[v2])` | Un par de variables |
| Matriz correlación | `df[cols].corr()` | Múltiples pares a la vez |
| Heatmap | `sns.heatmap(corr, annot=True)` | Visualizar matriz |

**Escala de interpretación de r:**

| Valor |r| | Interpretación |
|-------|---|----------------|
| 0.70 – 1.0 | Fuerte |
| 0.30 – 0.69 | Moderada |
| 0.00 – 0.29 | Débil o nula |

> 💡 **Correlación ≠ Causalidad.** Usa expresiones como *"se asocia con"* o *"existe una relación entre"* en lugar de *"causa"* o *"provoca"*.

> 💡 **Siempre visualiza antes de calcular Pearson.** Si la relación no es lineal (curva en U, campana), r puede ser ≈ 0 aunque haya una relación real.